The main goal of this project is to create a nlp model which can understand the human emotions using machine learning methods

In [3]:
import numpy as np
import pandas as pd

In [4]:
df=pd.read_csv('train.txt.zip',sep=';',header=None,names=['text','emotions'])
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [5]:
df.isnull().sum()

,0
text,0
emotions,0


In [6]:
df['emotions'].value_counts().sort_values(ascending=False)

,count
emotions,
joy,5362
sadness,4666
anger,2159
fear,1937
love,1304
surprise,572


###Encoding
Encoding the target values into numeric values

In [7]:
emotions_values=df['emotions'].unique()
mask={}
i=0
for emo in emotions_values:
  mask[emo]=i
  i+=1
df['emotions_no']=df['emotions'].map(mask)

In [43]:
emotions_values=df['emotions'].unique()
emotions_values

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [8]:
df

,text,emotions,emotions_no
0,i didnt feel humiliated,sadness,0
1,i can go from feeling so hopeless to so damned...,sadness,0
2,im grabbing a minute to post i feel greedy wrong,anger,1
3,i am ever feeling nostalgic about the fireplac...,love,2
4,i am feeling grouchy,anger,1
...,...,...,...
15995,i just had a very brief time in the beanbag an...,sadness,0
15996,i am now turning and i feel pathetic that i am...,sadness,0
15997,i feel strong and good overall,joy,5
15998,i feel like this was such a rude comment and i...,anger,1


###Data Cleaning
Vrious ways of cleaning the string data

In [9]:
#remove lowercase
df['text']=df['text'].apply(lambda x:x.lower())

In [10]:
#remove punchuations(*#$)
#str.maketrans('','',thing that you want to remove ('jshs')--->any chaacter)
#txt.translate(takes txt as input and apply that transformation)
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))
df['text']=df['text'].apply(remove_punc)

In [11]:
#removing numbers from text
def remove_number(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new
df['text']=df['text'].apply(remove_number)

###Removing Stepwords
Words that have no major use and should be removed as they increase noise, NLTK liabraries are used for this.

In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
stop_words=set(stopwords.words('english'))

In [14]:
df.loc[1,'text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [15]:
def remove_stopwords(txt):
  clean=[]
  split=word_tokenize(txt)
  for i in split:
    if i not in stop_words:
      clean.append(i)
  return ' '.join(clean)

In [16]:
df['text']=df['text'].apply(remove_stopwords)

In [17]:
df.loc[1,'text']

'go feeling hopeless damned hopeful around someone cares awake'

In [18]:
df

,text,emotions,emotions_no
0,didnt feel humiliated,sadness,0
1,go feeling hopeless damned hopeful around some...,sadness,0
2,im grabbing minute post feel greedy wrong,anger,1
3,ever feeling nostalgic fireplace know still pr...,love,2
4,feeling grouchy,anger,1
...,...,...,...
15995,brief time beanbag said anna feel like beaten,sadness,0
15996,turning feel pathetic still waiting tables sub...,sadness,0
15997,feel strong good overall,joy,5
15998,feel like rude comment im glad,anger,1


#Model Making

###Train Test Split

In [21]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df['text'],df['emotions_no'],test_size=0.2,random_state=42)

### Using Cross Validation and Pipeline making

In [36]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

pipeline={'Bow_Logistic':Pipeline([('vectorizer',CountVectorizer()),('model',LogisticRegression(max_iter=1000,class_weight='balanced'))]),'Bow_Naive':Pipeline([('vectorizer',CountVectorizer()),('model',MultinomialNB())]),
          'Tfidf_Logistic':Pipeline([('vectorizer',TfidfVectorizer()),('model',LogisticRegression(max_iter=1000,class_weight='balanced'))]),'Tfidf_Naive':Pipeline([('vectorizer',TfidfVectorizer()),('model',MultinomialNB())])}

for name,things in pipeline.items():
  scores=cross_val_score(things,x_train,y_train,cv=5,scoring='accuracy')
  print(f'{name}: {scores.mean():.4f}')

Bow_Logistic: 0.8860
Bow_Naive: 0.7616
Tfidf_Logistic: 0.8800
Tfidf_Naive: 0.6585


In [37]:
best_pipeline=pipeline['Bow_Logistic']
best_pipeline

Pipeline(steps=[('vectorizer', CountVectorizer()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

### Randomizesearch For best hyperparameter model

In [38]:
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'vectorizer__ngram_range': [(1,1), (1,2), (1,3)],
    'vectorizer__max_features': [2000, 3000, 5000, 8000],
    'vectorizer__min_df': [1, 2, 5],
    'vectorizer__max_df': [0.75, 0.85, 0.95],
    'vectorizer__stop_words': [None, 'english'],

    'model__C': [0.01, 0.1, 1, 10, 50],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs', 'liblinear']
}
search=RandomizedSearchCV(best_pipeline,param_distributions=param_dist,cv=5,n_jobs=-1,n_iter=20,scoring='accuracy',random_state=42)
search.fit(x_train,y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('vectorizer', CountVectorizer()),
                                             ('model',
                                              LogisticRegression(class_weight='balanced',
                                                                 max_iter=1000))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__C': [0.01, 0.1, 1, 10, 50],
                                        'model__penalty': ['l2'],
                                        'model__solver': ['lbfgs', 'liblinear'],
                                        'vectorizer__max_df': [0.75, 0.85,
                                                               0.95],
                                        'vectorizer__max_features': [2000, 3000,
                                                                     5000,
                                                                     8000],
                                        'vectorizer__min_df': [1, 2, 5],
                                        'vectorizer__ngram_range': [(1, 1),
                                                                    (1, 2),
                                                                    (1, 3)],
                                        'vectorizer__stop_words': [None,
                                                                   'english']},
                   random_state=42, scoring='accuracy')

In [39]:
best_model=search.best_estimator_

###Predicting with best model

In [40]:
from sklearn.metrics import accuracy_score,classification_report
y_pred=best_model.predict(x_test)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.904375
              precision    recall  f1-score   support

           0       0.95      0.92      0.94       946
           1       0.90      0.89      0.90       427
           2       0.80      0.90      0.85       296
           3       0.75      0.84      0.79       113
           4       0.87      0.87      0.87       397
           5       0.93      0.92      0.93      1021

    accuracy                           0.90      3200
   macro avg       0.87      0.89      0.88      3200
weighted avg       0.91      0.90      0.91      3200



###Saving Model

In [41]:
import joblib
joblib.dump(best_model,'emotion_classifier.joblib')
from google.colab import files
files.download('emotion_classifier.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>